# Step 10-B2: kiểm định ghép cặp + Holm cho hai LLM chấm

So QuDAR-llm khi chấm bằng gpt-4o-mini (Step 4, 6) và Qwen2.5-7B-Instruct (Step 10-B1), theo 4 họ kiểm định đặt trước (Holm trong từng họ):
F1 QuDAR-llm so với Equal; F2 Qwen so với gpt-4o-mini; F3 ablation 2×2 (thứ tự × prompt); F4 thiên lệch vị trí (kiểm định hoán vị).

- **Accelerator:** None (CPU). **Internet:** On (clone repo để lấy điểm gpt-4o-mini trong `results/`).
- **Input:** `qudar-runs`, `qudar-beir-data`, `qudar-llm-qwen`, `qudar-patches` (bản mới có `judge_significance.py`). Không cần gắn `qudar-llm` hay `qudar-llm-ablation`.
- **Thời gian ước tính:** cài đặt 1–2 phút; đọc `retrieval_result.json` của HotpotQA 1–3 phút; tính chỉ số từng query khoảng 2 phút; kiểm định 3–5 phút. Tổng khoảng 10–15 phút.
- **Sau khi chạy:** Save Version → tải các file trong `/kaggle/working/step10b2/` về repo, đặt vào `results/step10/sig/`.

In [1]:
import glob, os
DATA  = "/kaggle/input/datasets/karrychuong/qudar-beir-data/data"
PATCH = os.path.dirname(glob.glob("/kaggle/input/**/judge_significance.py", recursive=True)[0])
RUNS  = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/fiqa_test/retrieval_result.json", recursive=True)[0]))
qwen  = [p for p in glob.glob("/kaggle/input/**/llm_scores_paper_fiqa_test.csv", recursive=True) if "qwen" in p.lower()]
assert len(qwen) == 1, f"cần đúng 1 thư mục điểm Qwen, thấy: {qwen}"
QWEN  = os.path.dirname(qwen[0])
assert os.path.isdir(DATA) and os.path.exists(f"{PATCH}/grid_sweep.py")
!git clone -q --depth 1 https://github.com/KarryChuong/CS419-IR-QuDAR-replication.git /kaggle/working/repo
GPT = "/kaggle/working/repo/results/step4_llm/test,/kaggle/working/repo/results/step6_ablation"
!pip install -q tabulate
print("PATCH =", PATCH, "\nRUNS  =", RUNS, sorted(os.listdir(RUNS)), "\nQWEN  =", QWEN, "\nGPT   =", GPT)

PATCH = /kaggle/input/datasets/karrychuong/qudar-patches/qudar-patches 
RUNS  = /kaggle/input/datasets/karrychuong/qudar-runs/runs ['climate-fever_test', 'fiqa_test', 'fiqa_train', 'hotpotqa_test', 'hotpotqa_train', 'scidocs_test'] 
QWEN  = /kaggle/input/datasets/karrychuong/qudar-llm-qwen/llm_qwen 
GPT   = /kaggle/working/repo/results/step4_llm/test,/kaggle/working/repo/results/step6_ablation


In [2]:
!python {PATCH}/judge_significance.py --runs {RUNS} --data-root {DATA} \
    --judge gpt-4o-mini={GPT} --judge Qwen2.5-7B={QWEN} --out /kaggle/working/step10b2

[12:02:23] fiqa_test: loading retrieval_result.json
[12:02:29] fiqa_test: 648 queries, 8 LLM runs
[12:02:29] scidocs_test: loading retrieval_result.json
[12:02:38] scidocs_test: 1000 queries, 8 LLM runs
[12:02:39] climate-fever_test: loading retrieval_result.json
[12:02:56] climate-fever_test: 1535 queries, 8 LLM runs
[12:02:57] hotpotqa_test: loading retrieval_result.json
[12:03:41]   hotpotqa_test: 2000/7405 queries
[12:03:45]   hotpotqa_test: 4000/7405 queries
[12:03:49]   hotpotqa_test: 6000/7405 queries
[12:03:52] hotpotqa_test: 7405 queries, 2 LLM runs
[12:03:56] hotpotqa_test_first1500: 1500 queries, 8 LLM runs

Check: recomputed means vs reported (max |diff| should be <= 0.0001, rounding)
                    set                   method  ours Recall@10  reported  max |diff|
              fiqa_test                    Equal          0.3333    0.3333      0.0000
              fiqa_test        gpt-4o-mini|paper          0.3281    0.3281      0.0000
              fiqa_test  gpt-4o-m

In [3]:
# Bản gọn để gửi lại: chênh lệch A - B theo từng bộ, dấu * = có ý nghĩa sau Holm
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 50)
OUT = "/kaggle/working/step10b2"
t = pd.read_csv(f"{OUT}/sig_tests.csv")
t["cell"] = t["mean(A-B)"].map("{:+.4f}".format) + t["sig (Holm p<0.05)"].fillna("")
cols = list(dict.fromkeys(t["set"]))
pv = t.pivot_table(index=["family", "judge", "comparison", "metric"], columns="set", values="cell",
                   aggfunc="first", sort=False)[cols]
print(pv.to_string())
pos = pd.read_csv(f"{OUT}/position_tests.csv")
print("\n" + pos[["judge", "set", "mode", "queries", "pos 1", "pos 2", "pos 3", "pos 4", "p Holm", "sig (Holm p<0.05)"]].to_string(index=False))
chk = pd.read_csv(f"{OUT}/check_means.csv")
print("\nĐối chiếu với số đã báo cáo: max |diff| =", chk["max |diff|"].max())
print(open(f"{OUT}/summary.json").read())

set                                                                                                                   FiQA  SciDocs Climate-FEVER  HotpotQA HotpotQA (first 1500)
family          judge                    comparison                                                    metric                                                                    
F1 llm_vs_equal gpt-4o-mini              QuDAR-llm (paper prompt + shuffled) - Equal                   Recall@10   -0.0053  -0.0002       +0.0063  -0.0123*                   NaN
                                                                                                       nDCG@10     +0.0027  +0.0006       +0.0046   +0.0037                   NaN
                                         QuDAR-llm (code prompt + shuffled) - Equal                    Recall@10   +0.0053  +0.0019      +0.0105*       NaN               +0.0037
                                                                                                       nDCG@10